In [ ]:
import pandas as pd
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
colunas = [
    'causa_acidente',
    'tipo_acidente',
    'fase_dia',
    'condicao_metereologica',
    'tipo_pista',
    'tracado_via',
    'dia_semana',
    'uf',
    'br',
    'uso_solo',
    'pessoas',
    'veiculos',
    'mortos'
]

dados puros

In [ ]:
pasta = '/content/drive/MyDrive/Machine Learning/data/'
anos = range(2018, 2026)
frames = []

for ano in anos:
    caminho = os.path.join(pasta, f'datatran{ano}.csv')

    if not os.path.exists(caminho):
        print(f'⚠️  Arquivo {ano} não encontrado, pulando...')
        continue

    df = pd.read_csv(caminho, encoding='latin-1', sep=';', usecols=colunas, low_memory=False)
    frames.append(df)
    print(f'✅ {ano} — {len(df)} linhas')

df_raw = pd.concat(frames, ignore_index=True)

print(f'\nTotal: {len(df_raw)} linhas')
print(df_raw['mortos'].value_counts())

saida = os.path.join(pasta, 'prf_puro.parquet')
df_raw.to_parquet(saida, index=False)
print(f'\n💾 Salvo em: {saida}')

✅ 2018 — 69333 linhas
✅ 2019 — 67558 linhas
✅ 2020 — 63585 linhas
✅ 2021 — 64567 linhas
✅ 2022 — 64606 linhas
✅ 2023 — 67766 linhas
✅ 2024 — 73156 linhas
✅ 2025 — 72529 linhas

Total: 543100 linhas
mortos
0     504849
1      33853
2       3280
3        698
4        257
5         98
6         29
7         15
8          5
9          3
12         3
11         2
13         2
19         1
23         1
14         1
37         1
16         1
10         1
Name: count, dtype: int64

💾 Salvo em: /content/drive/MyDrive/Machine Learning/data/prf_puro.parquet


dados balançeados e normalizados

In [ ]:
pasta = '/content/drive/MyDrive/Machine Learning/data/'
anos = range(2018, 2026)
frames = []

for ano in anos:
    caminho = os.path.join(pasta, f'datatran{ano}.csv')

    if not os.path.exists(caminho):
        print(f'⚠️  Arquivo {ano} não encontrado, pulando...')
        continue

    df = pd.read_csv(caminho, encoding='latin-1', sep=';', usecols=colunas, low_memory=False)

    df['mortos'] = (df['mortos'] >= 1).astype(int)

    com_mortos    = df[df['mortos'] == 1]
    sem_mortos    = df[df['mortos'] == 0]

    n = len(com_mortos)
    sem_mortos_amostra = sem_mortos.sample(n=n, random_state=42)

    df_balanceado = pd.concat([com_mortos, sem_mortos_amostra], ignore_index=True)
    frames.append(df_balanceado)

    print(f'✅ {ano} — {n} fatais + {n} não fatais = {len(df_balanceado)} linhas')

df_final = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=42)

print(f'\nTotal final: {len(df_final)} linhas')
print(df_final['mortos'].value_counts())

saida = os.path.join(pasta, 'prf_balanceado.parquet')
df_final.to_parquet(saida, index=False)
print(f'\n💾 Salvo em: {saida}')

✅ 2018 — 4508 fatais + 4508 não fatais = 9016 linhas
✅ 2019 — 4599 fatais + 4599 não fatais = 9198 linhas
✅ 2020 — 4525 fatais + 4525 não fatais = 9050 linhas
✅ 2021 — 4665 fatais + 4665 não fatais = 9330 linhas
✅ 2022 — 4664 fatais + 4664 não fatais = 9328 linhas
✅ 2023 — 4858 fatais + 4858 não fatais = 9716 linhas
✅ 2024 — 5222 fatais + 5222 não fatais = 10444 linhas
✅ 2025 — 5210 fatais + 5210 não fatais = 10420 linhas

Total final: 76502 linhas
mortos
0    38251
1    38251
Name: count, dtype: int64

💾 Salvo em: /content/drive/MyDrive/Machine Learning/data/prf_balanceado.parquet
